In [ ]:
from pathlib import Path
import sys


def find_curtailment_scripts_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        direct = candidate / "SAPN2022_Analysis" / "curtailment" / "scripts"
        if (direct / "path_config.py").exists():
            return direct
        if (candidate / "path_config.py").exists() and candidate.name == "scripts":
            return candidate
    raise RuntimeError(
        "Could not locate SAPN2022_Analysis/curtailment/scripts from the current working directory."
    )


SCRIPTS_DIR = find_curtailment_scripts_dir(Path.cwd().resolve())
PROJECT_ROOT = SCRIPTS_DIR.parent

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

# Local SAPN/BOM paths live in `local_paths.py` so this tracked notebook does
# not commit machine-specific local path values. Start from
# `local_paths.example.py` if you need to create the local file.
from path_config import require_local_path

PHASE_B_PATH = require_local_path(
    "PHASE_B_TIMESTAMP_DETAIL_PATH",
    "Phase B timestamp detail CSV used by the timestamp debugging notebook.",
)
PHASE_B_ALT_PATH = require_local_path(
    "PHASE_B_TIMESTAMP_DETAIL_ALT_PATH",
    "alternate Phase B timestamp detail CSV used by the comparison cells.",
)
SAPN_ROOT = require_local_path(
    "SAPN_ROOT",
    "root folder containing the external `Nov2022/` SAPN exports.",
)
UNCURTAILED_PATH = PROJECT_ROOT / "outputs" / "local_scored" / "all_uncurtailedPV.parquet"
        


In [ ]:
import polars as pl

phase_b_path = PHASE_B_PATH

df = pl.read_csv(phase_b_path)

# parse timestamp (keeps timezone info)
df = df.with_columns(
    pl.col("local_tstamp")
    .str.strptime(
        pl.Datetime,
        "%Y-%m-%dT%H:%M:%S%.f%z",
        strict=False
    )
)

# create both views
df = df.with_columns([
    # convert to UTC (naive)
    pl.col("local_tstamp")
    .dt.convert_time_zone("UTC")
    .dt.replace_time_zone(None)
    .alias("t_stamp_utc"),

    # explicitly keep Adelaide local time
    pl.col("local_tstamp")
    .dt.convert_time_zone("Australia/Adelaide")
    .alias("local_tstamp_adelaide"),
])

# print min/max for both
result = df.select([
    pl.col("local_tstamp_adelaide").min().alias("min_local"),
    pl.col("local_tstamp_adelaide").max().alias("max_local"),
    pl.col("t_stamp_utc").min().alias("min_utc"),
    pl.col("t_stamp_utc").max().alias("max_utc"),
])

print(result)
    


In [ ]:
import polars as pl

site_id = None  # set to a real site_id before running

if site_id is None:
    raise ValueError("Set site_id before running this cell.")

# -----------------------
# 1. all_uncurtailedPV
# -----------------------
uncurtailed_path = UNCURTAILED_PATH

uncurtailed = (
    pl.read_parquet(uncurtailed_path)
    .filter(pl.col("site_id") == site_id)
    .with_columns(
        pl.col("t_stamp")
        .dt.replace_time_zone("UTC")
        .dt.convert_time_zone("Australia/Adelaide")
        .alias("local_tstamp")
    )
    .filter(
        (pl.col("local_tstamp").dt.hour() >= 6) &
        (pl.col("local_tstamp").dt.hour() < 18) &
        (pl.col("local_tstamp").dt.day().is_in([13, 14, 15, 16, 17, 19]))

    )
    .select(["site_id", "t_stamp"])
    .unique()
)

print("Uncurtailed (6–18h) rows:", uncurtailed.shape[0])

# -----------------------
# 2. Phase B
# -----------------------
phase_b_path = PHASE_B_ALT_PATH

phase_b = (
    pl.read_csv(phase_b_path)
    .filter(pl.col("site_id") == site_id)
    .with_columns(
        pl.col("local_tstamp")
        .str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M:%S%.f%z", strict=False)
    )
    .with_columns([
        # keep local for filtering
        pl.col("local_tstamp")
        .dt.convert_time_zone("Australia/Adelaide")
        .alias("local_tstamp_adl"),

        # convert to UTC for joining
        pl.col("local_tstamp")
        .dt.convert_time_zone("UTC")
        .dt.replace_time_zone(None)
        .alias("t_stamp"),
    ])
    .filter(
        (pl.col("local_tstamp_adl").dt.hour() >= 6) &
        (pl.col("local_tstamp_adl").dt.hour() < 18) &
        (pl.col("local_tstamp_adl").dt.day().is_in([13, 14, 15, 16, 17, 19]))
    )
    .select(["site_id", "t_stamp"])
    .unique()
)

print("Phase B (6–18h) rows:", phase_b.shape[0])

# -----------------------
# 3. Compare
# -----------------------

matched = uncurtailed.join(phase_b, on=["site_id", "t_stamp"], how="inner")

missing_in_phase_b = uncurtailed.join(
    phase_b, on=["site_id", "t_stamp"], how="anti"
)

missing_in_uncurtailed = phase_b.join(
    uncurtailed, on=["site_id", "t_stamp"], how="anti"
)

print("
--- Comparison (6–18h only) ---")
print("Matched:", matched.shape[0])
print("Missing in Phase B:", missing_in_phase_b.shape[0])
print("Missing in Uncurtailed:", missing_in_uncurtailed.shape[0])
    


In [ ]:
missing = uncurtailed.join(
    phase_b,
    on=["site_id", "t_stamp"],
    how="anti"
)

print("Missing count:", missing.shape[0])

In [ ]:
# compare uncurtaield timestamps with original source
import polars as pl

site_id = None  # set to a real site_id before running

if site_id is None:
    raise ValueError("Set site_id before running this cell.")
mapping_path = SAPN_ROOT / "Nov2022" / "ebm_1_20221112_20221119_circuit_details.csv"
source_path = SAPN_ROOT / "Nov2022" / "ebm_1_20221112_20221119_data_processed_sa.parquet"
uncurtailed_path = UNCURTAILED_PATH

# -----------------------
# 1. Source data
# -----------------------

site_circuits = (
    pl.read_csv(mapping_path)
    .filter(pl.col("site_id") == site_id)
    .select("c_id")
    .unique()
)

c_ids = site_circuits.get_column("c_id").to_list()

source_with_site = (
    pl.scan_parquet(source_path)
    .filter(pl.col("c_id").is_in(c_ids))
    .select([
        pl.lit(site_id).alias("site_id"),
        pl.col("utc_tstamp")
        .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S%.f", strict=False)
        .alias("t_stamp"),
    ])
    .unique()
)

# -----------------------
# 2. Uncurtailed data
# -----------------------
uncurtailed = (
    pl.scan_parquet(uncurtailed_path)
    .filter(pl.col("site_id") == site_id)
    .select(["site_id", "t_stamp"])
    .unique()
)

# -----------------------
# 3. Compare
# -----------------------

present_in_source = uncurtailed.join(
    source_with_site,
    on=["site_id", "t_stamp"],
    how="inner"
).collect()

missing_in_source = uncurtailed.join(
    source_with_site,
    on=["site_id", "t_stamp"],
    how="anti"
).collect()

uncurtailed_total = uncurtailed.select(pl.len().alias("n")).collect().item()

print("Mapped circuits:", len(c_ids))
print("Uncurtailed total:", uncurtailed_total)
print("Present in source:", present_in_source.shape[0])
print("Missing in source:", missing_in_source.shape[0])
    
